## Libraries

In [ ]:
import os
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import ruptures as rpt


## Data Preprocessing and Alignment

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def load_e4_csv(file_path: Path) -> tuple[float, float, np.ndarray]:
    """Load physiological signal from an Empatica E4 CSV file.

    The Empatica E4 CSV format consists of:
    - Line 1: Start timestamp (POSIX UTC float)
    - Line 2: Sampling frequency (Hz float)
    - Line 3 onwards: Raw signal values

    Parameters
    ----------
    file_path : Path
        Path to the E4 CSV file.

    Returns
    -------
    tuple[float, float, np.ndarray]
        A tuple containing (start_time, frequency, signal_values).
    """
    with open(file_path, "r", encoding="utf-8") as f:
        start_time = float(f.readline().strip())
        frequency = float(f.readline().strip())

    signal_values = np.loadtxt(file_path, skiprows=2, delimiter=",")
    return start_time, frequency, signal_values


def align_signals(
    eda_data: tuple[float, float, np.ndarray],
    temp_data: tuple[float, float, np.ndarray],
    hr_data: tuple[float, float, np.ndarray],
    target_freq: float = 4.0,
) -> tuple[float, pd.DataFrame]:
    """Upsample and temporally align EDA, TEMP, and HR signals to a target frequency.

    The signals are upsampled by sample repetition to match the target frequency,
    then aligned to the maximum of the start times and cropped to the minimum
    of the end times.

    Parameters
    ----------
    eda_data : tuple[float, float, np.ndarray]
        (start_time, frequency, values) for Electrodermal Activity.
    temp_data : tuple[float, float, np.ndarray]
        (start_time, frequency, values) for Skin Temperature.
    hr_data : tuple[float, float, np.ndarray]
        (start_time, frequency, values) for Heart Rate.
    target_freq : float, default 4.0
        The frequency (Hz) to align the signals to.

    Returns
    -------
    tuple[float, pd.DataFrame]
        A tuple containing:
        - reference_time (float): The timestamp of the first sample in the dataframe.
        - df_aligned (pd.DataFrame): Dataframe with columns ['EDA', 'HR', 'temp'].
    """
    eda_start, eda_freq, eda_vals = eda_data
    temp_start, temp_freq, temp_vals = temp_data
    hr_start, hr_freq, hr_vals = hr_data

    repeat_eda = int(round(target_freq / eda_freq))
    eda_upsampled = np.repeat(eda_vals, repeat_eda)

    repeat_temp = int(round(target_freq / temp_freq))
    temp_upsampled = np.repeat(temp_vals, repeat_temp)

    repeat_hr = int(round(target_freq / hr_freq))
    hr_upsampled = np.repeat(hr_vals, repeat_hr)

    start_time = max(eda_start, temp_start, hr_start)

    eda_end = eda_start + (len(eda_upsampled) - 1) / target_freq
    temp_end = temp_start + (len(temp_upsampled) - 1) / target_freq
    hr_end = hr_start + (len(hr_upsampled) - 1) / target_freq
    end_time = min(eda_end, temp_end, hr_end)

    eda_idx_start = int(round((start_time - eda_start) * target_freq))
    temp_idx_start = int(round((start_time - temp_start) * target_freq))
    hr_idx_start = int(round((start_time - hr_start) * target_freq))

    num_samples = int(round((end_time - start_time) * target_freq)) + 1

    eda_aligned = eda_upsampled[eda_idx_start : eda_idx_start + num_samples]
    temp_aligned = temp_upsampled[temp_idx_start : temp_idx_start + num_samples]
    hr_aligned = hr_upsampled[hr_idx_start : hr_idx_start + num_samples]

    df_aligned = pd.DataFrame(
        {
            "EDA": eda_aligned,
            "HR": hr_aligned,
            "temp": temp_aligned,
        }
    )

    return start_time, df_aligned


def create_lag_features(
    df_features: pd.DataFrame, lag_steps: int = 10
) -> pd.DataFrame:
    """Construct lag features for HR_Mean, temp_Mean, and EDA_Mean.

    Parameters
    ----------
    df_features : pd.DataFrame
        Dataframe containing the computed window features.
    lag_steps : int, default 10
        Number of steps to shift back.

    Returns
    -------
    pd.DataFrame
        Dataframe containing the 30 lag columns (named '30' down to '1')
        aligned with df_features (first lag_steps rows will have NaNs).
    """
    cols = [str(x) for x in range(30, 0, -1)]
    lag_cols = []

    for step in range(lag_steps, 0, -1):
        lag_cols.append(df_features["HR_Mean"].shift(step))
    for step in range(lag_steps, 0, -1):
        lag_cols.append(df_features["temp_Mean"].shift(step))
    for step in range(lag_steps, 0, -1):
        lag_cols.append(df_features["EDA_Mean"].shift(step))

    df_lags = pd.concat(lag_cols, axis=1)
    df_lags.columns = cols
    return df_lags


## Feature Extraction

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.stats import kurtosis, skew


def statistical_features(arr: np.ndarray) -> tuple[float, float, float, float]:
    """Compute descriptive statistics for a 1D numeric array.

    Parameters
    ----------
    arr : np.ndarray
        One-dimensional array of numeric values.

    Returns
    -------
    tuple[float, float, float, float]
        Minimum, maximum, mean, and standard deviation of the array.
    """
    return float(np.amin(arr)), float(np.amax(arr)), float(np.mean(arr)), float(np.std(arr))


def shape_features(arr: np.ndarray) -> tuple[float, float]:
    """Compute skewness and kurtosis for a 1D numeric array.

    Parameters
    ----------
    arr : np.ndarray
        One-dimensional array of numeric values.

    Returns
    -------
    tuple[float, float]
        Skewness and kurtosis of the array.
    """
    return float(skew(arr)), float(kurtosis(arr))


def compute_rmssd(arr: np.ndarray) -> float:
    """Compute Root Mean Square of Successive Differences (RMSSD) for a 1D array.

    Parameters
    ----------
    arr : np.ndarray
        One-dimensional array of numeric values.

    Returns
    -------
    float
        RMSSD of the array.
    """
    diffs = np.ediff1d(arr)
    if len(diffs) == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(diffs))))


def compute_eda_peaks(arr: np.ndarray, min_width: float = 5.0) -> tuple[int, float, float]:
    """Detect peaks in Electrodermal Activity (EDA) signal.

    Finds peaks using scipy's find_peaks and computes the number of peaks,
    the sum of peak prominences, and the sum of peak widths.

    Parameters
    ----------
    arr : np.ndarray
        One-dimensional array of EDA values.
    min_width : float, default 5.0
        Minimum width threshold for peak detection.

    Returns
    -------
    tuple[int, float, float]
        A tuple containing (num_peaks, sum_prominences, sum_widths).
    """
    peaks, properties = find_peaks(arr, width=min_width)
    num_peaks = len(peaks)
    prominences = np.array(properties.get("prominences", [0.0]))
    widths = np.array(properties.get("widths", [0.0]))
    amplitude = float(np.sum(prominences))
    duration = float(np.sum(widths))
    return num_peaks, amplitude, duration


def extract_window_features(
    df_aligned: pd.DataFrame, window_size: int = 40, step: int = 20
) -> pd.DataFrame:
    """Extract physiological features over rolling windows.

    Parameters
    ----------
    df_aligned : pd.DataFrame
        Dataframe with columns ['EDA', 'HR', 'temp'].
    window_size : int, default 40
        Size of the rolling window in samples (e.g. 40 samples at 4Hz = 10s).
    step : int, default 20
        Step size between windows in samples (e.g. 20 samples = 50% overlap).

    Returns
    -------
    pd.DataFrame
        Dataframe containing 18 extracted features for each window.
    """
    cols = [
        "EDA_Mean", "EDA_Min", "EDA_Max", "EDA_Std", "EDA_Kurtosis", "EDA_Skew",
        "EDA_Num_Peaks", "EDA_Amphitude", "EDA_Duration",
        "HR_Mean", "HR_Min", "HR_Max", "HR_Std", "HR_RMS",
        "temp_Mean", "temp_Min", "temp_Max", "temp_Std"
    ]

    features_list = []
    eda_vals = df_aligned["EDA"].to_numpy()
    hr_vals = df_aligned["HR"].to_numpy()
    temp_vals = df_aligned["temp"].to_numpy()

    for i in range(0, len(df_aligned), step):
        if i + window_size > len(df_aligned):
            break

        eda_win = eda_vals[i : i + window_size]
        hr_win = hr_vals[i : i + window_size]
        temp_win = temp_vals[i : i + window_size]

        eda_min, eda_max, eda_mean, eda_std = statistical_features(eda_win)
        hr_min, hr_max, hr_mean, hr_std = statistical_features(hr_win)
        temp_min, temp_max, temp_mean, temp_std = statistical_features(temp_win)

        eda_skew, eda_kurt = shape_features(eda_win)
        hr_rms = compute_rmssd(hr_win)
        num_peaks, amplitude, duration = compute_eda_peaks(eda_win)

        features_list.append([
            eda_mean, eda_min, eda_max, eda_std, eda_kurt, eda_skew,
            num_peaks, amplitude, duration,
            hr_mean, hr_min, hr_max, hr_std, hr_rms,
            temp_mean, temp_min, temp_max, temp_std
        ])

    return pd.DataFrame(features_list, columns=cols)


## Model Training and Evaluation

In [ ]:
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


def evaluate_model_cv(
    model, X: pd.DataFrame, y: pd.Series, cv_splits: int = 5
) -> list[float]:
    """Perform TimeSeriesSplit cross-validation on a model.

    Parameters
    ----------
    model : estimator
        Scikit-learn estimator.
    X : pd.DataFrame
        Feature matrix.
    y : pd.Series
        Target vector.
    cv_splits : int, default 5
        Number of splits for cross-validation.

    Returns
    -------
    list[float]
        Accuracy scores for each fold.
    """
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    scores = []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train.to_numpy().ravel())
        score = model.score(X_test, y_test.to_numpy().ravel())
        scores.append(score)

    return scores


def train_and_evaluate_all(
    features: pd.DataFrame, labels: pd.DataFrame, test_size_ratio: float = 0.33
) -> dict[str, object]:
    """Train and evaluate RF, KNN, and SVM models using a chronological split.

    Parameters
    ----------
    features : pd.DataFrame
        The feature matrix (first 48 columns).
    labels : pd.DataFrame
        The target label (stress column).
    test_size_ratio : float, default 0.33
        The ratio of data to use for chronological testing.

    Returns
    -------
    dict[str, object]
        Dictionary of trained models: {'RF': rf_model, 'KNN': knn_model, 'SVM': svm_model}.
    """
    total_len = len(features)
    split_idx = int(round(total_len * (1 - test_size_ratio)))

    X_train, X_test = features.iloc[:split_idx], features.iloc[split_idx:]
    y_train, y_test = labels.iloc[:split_idx], labels.iloc[split_idx:]

    y_train_flat = y_train.to_numpy().ravel()
    y_test_flat = y_test.to_numpy().ravel()

    models = {
        "RF": RandomForestClassifier(n_estimators=150, max_depth=15, class_weight="balanced", random_state=30),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "SVM": SVC(class_weight="balanced", random_state=30),
    }

    trained_models = {}

    for name, model in models.items():
        print(f"=== Evaluating {name} ===")
        cv_scores = evaluate_model_cv(model, features, labels)
        print(f"TimeSeriesSplit CV Accuracy: {sum(cv_scores)/len(cv_scores):.4f} {cv_scores}")

        model.fit(X_train, y_train_flat)
        y_pred = model.predict(X_test)

        print("Chronological Test Set Classification Report:")
        print(classification_report(y_test_flat, y_pred, zero_division=0))
        print("\n")

        trained_models[name] = model

    return trained_models


## Change Point Detection and Segmentation

In [ ]:
from datetime import datetime
import pandas as pd
import ruptures as rpt


def detect_change_points(predictions: pd.Series, penalty: float = 10.0) -> list[int]:
    """Run Pelt change point detection on predicted stress labels.

    Parameters
    ----------
    predictions : pd.Series
        Series of predicted stress labels.
    penalty : float, default 10.0
        The penalty parameter for Pelt search.

    Returns
    -------
    list[int]
        List of change point indices, starting with 0.
    """
    signal = predictions.to_numpy().reshape(-1)
    algo = rpt.Pelt(model="l2").fit(signal)
    result = algo.predict(pen=penalty)
    return [0] + result


def classify_and_merge_segments(
    df_total: pd.DataFrame,
    change_points: list[int],
    reference_time: float,
    window_step_seconds: float = 5.0,
) -> pd.DataFrame:
    """Classify and merge segments of stress labels using segment-wide averages.

    Parameters
    ----------
    df_total : pd.DataFrame
        Dataframe containing aligned features and the 'pred' column.
    change_points : list[int]
        List of change point indices.
    reference_time : float
        POSIX timestamp corresponding to the start of the aligned time series.
    window_step_seconds : float, default 5.0
        Seconds per step (e.g. 20 samples at 4Hz = 5s).

    Returns
    -------
    pd.DataFrame
        Dataframe with merged stress intervals: ['start_time', 'end_time', 'duration', 'stress_level'].
    """
    stress_labels = []

    for i in range(len(change_points) - 1):
        segment = df_total.iloc[change_points[i] : (change_points[i + 1] - 1)]
        if not segment.empty:
            mean_stress = segment["pred"].mean()
        else:
            mean_stress = 0.0

        if mean_stress > 1.3:
            level = 2
        elif mean_stress >= 0.65:
            level = 1
        else:
            level = 0

        stress_labels.append(level)

    temp_intervals = []
    for i in range(len(change_points) - 1):
        temp_intervals.append(
            {
                "start": change_points[i],
                "end": change_points[i + 1],
                "stress": stress_labels[i],
            }
        )

    merged_intervals = []
    if not temp_intervals:
        return pd.DataFrame(columns=["start_time", "end_time", "duration", "stress_level"])

    stress_start = temp_intervals[0]["start"]
    stress_end = temp_intervals[0]["end"]
    previous_stress = temp_intervals[0]["stress"]

    for item in temp_intervals[1:]:
        if item["stress"] == previous_stress:
            stress_end = item["end"]
        else:
            start_dt = datetime.fromtimestamp(reference_time + (stress_start * window_step_seconds))
            end_dt = datetime.fromtimestamp(reference_time + (stress_end * window_step_seconds))
            duration = end_dt - start_dt

            merged_intervals.append(
                {
                    "start_time": start_dt.strftime("%H:%M:%S"),
                    "end_time": end_dt.strftime("%H:%M:%S"),
                    "duration": str(duration),
                    "stress_level": float(previous_stress),
                }
            )
            stress_start = item["start"]
            stress_end = item["end"]
            previous_stress = item["stress"]

    start_dt = datetime.fromtimestamp(reference_time + (stress_start * window_step_seconds))
    end_dt = datetime.fromtimestamp(reference_time + (stress_end * window_step_seconds))
    duration = end_dt - start_dt

    merged_intervals.append(
        {
            "start_time": start_dt.strftime("%H:%M:%S"),
            "end_time": end_dt.strftime("%H:%M:%S"),
            "duration": str(duration),
            "stress_level": float(previous_stress),
        }
    )

    return pd.DataFrame(merged_intervals)


## Run Pipeline

In [ ]:
# Load training set
base_dir = Path(".")
train_data_path = base_dir / "combined_lagEDA.csv"
df_lag = pd.read_csv(train_data_path)
X_train_full = df_lag.iloc[:, 0:48]
y_train_full = df_lag.iloc[:, 48:49]

# Train RF, KNN, and SVM models chronologically
trained_models = train_and_evaluate_all(X_train_full, y_train_full)

# Subject and raw files
subject = "DF"
subject_dir = base_dir / subject
eda_data = load_e4_csv(subject_dir / "EDA.csv")
temp_data = load_e4_csv(subject_dir / "TEMP.csv")
hr_data = load_e4_csv(subject_dir / "HR.csv")

# Preprocess and Align E4 Data
ref_time, df_aligned = align_signals(eda_data, temp_data, hr_data, target_freq=4.0)
df_features = extract_window_features(df_aligned, window_size=40, step=20)
df_lag_features = create_lag_features(df_features, lag_steps=10)
df_total = pd.concat([df_lag_features, df_features], axis=1).dropna()

# Scale Features
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(df_total.iloc[:, 0:48])
df_scaled = pd.DataFrame(x_scaled, columns=df_total.columns[:48]).fillna(0.0)

# Map feature names to training feature names
feature_names_mapping = {
    "EDA_Mean": "EDAR_Mean",
    "EDA_Min": "EDAR_Min",
    "EDA_Max": "EDAR_Max",
    "EDA_Std": "EDAR_Std",
    "EDA_Kurtosis": "EDAR_Kurtosis",
    "EDA_Skew": "EDAR_Skew",
    "EDA_Num_Peaks": "Num_PeaksR",
    "EDA_Amphitude": "EDAR_Amphitude",
    "EDA_Duration": "EDAR_Duration",
    "HR_Mean": "HRR_Mean",
    "HR_Min": "HRR_Min",
    "HR_Max": "HRR_Max",
    "HR_Std": "HRR_Std",
    "HR_RMS": "HRR_RMS",
    "temp_Mean": "TEMPR_Mean",
    "temp_Min": "TEMPR_Min",
    "temp_Max": "TEMPR_Max",
    "temp_Std": "TEMPR_Std",
}
df_scaled = df_scaled.rename(columns=feature_names_mapping)
df_scaled = df_scaled[X_train_full.columns]

# Predict and save RandomForest predictions
rf_model = trained_models["RF"]
rf_preds = rf_model.predict(df_scaled)
df_total_rf = df_total.copy()
df_total_rf["pred"] = rf_preds
df_total_rf.to_csv(base_dir / f"pred{subject}.csv", index=False)

# Predict and save SVM predictions
svm_model = trained_models["SVM"]
svm_preds = svm_model.predict(df_scaled)
df_total_svm = df_total.copy()
df_total_svm["pred"] = svm_preds
df_total_svm.to_csv(base_dir / f"predSVM{subject}.csv", index=False)

# Change point detection on predictions
change_points = detect_change_points(df_total_rf["pred"], penalty=10.0)
print(f"Detected change point indices: {change_points}")

# Plot change points Pelt Search Method
signal = df_total_rf["pred"].values.reshape(-1)
rpt.display(signal, change_points, figsize=(16, 6))
plt.title("Change Point Detection: Pelt Search Method")
plt.show()

# Segment and output stress events chronologically
df_events = classify_and_merge_segments(
    df_total_rf, change_points, reference_time=ref_time, window_step_seconds=5.0
)

print("\n=== Detected Stress Event Intervals ===")
for _, row in df_events.iterrows():
    print(
        f"Interval: {row['start_time']} - {row['end_time']} | "
        f"Duration: {row['duration']} | "
        f"Stress Level: {row['stress_level']}"
    )
